In [51]:
from tensorflow import keras # type: ignore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from datetime import datetime
import os
import pandas as pd
import numpy as np

In [52]:
#Opens a new dataframe with the Clean csv
cleancsv = pd.read_csv('CSV/CLEAN.csv')

In [53]:
#Convert data into Date time and create date filter
cleancsv['Date'] = pd.to_datetime(cleancsv['Date'])
cleancsv['Date'] = cleancsv['Date'] + pd.to_timedelta(cleancsv["Hr"], unit="h")
cleancsv.drop('Hr', axis=1, inplace=True)

"""
Use this in future if data set needs specific dates
prediction = data.loc{
    (untouched_csv['Date'] > datetime(x, x, x)) &
    (untouched_csv['Date'] < datetime(x, x, x,))
}
"""

"\nUse this in future if data set needs specific dates\nprediction = data.loc{\n    (untouched_csv['Date'] > datetime(x, x, x)) &\n    (untouched_csv['Date'] < datetime(x, x, x,))\n}\n"

In [54]:
#Create month colomn and restrict to only summer months
summer_mask = cleancsv["Date"].dt.month.isin([6, 7, 8])
cleancsv = cleancsv[summer_mask].reset_index(drop=True)

In [55]:
#Prepare colomns into variables
data_main_air_temp = cleancsv['Mainland Air Temp']
data_humidity_per = cleancsv['Humidity (%)']
data_wind_direction = cleancsv['Direction (A)']
data_wind_speed = cleancsv['Wind Speed (A)']
data_gusting = cleancsv['Gusting']
data_pressure = cleancsv['Atmospheric Pressure (IN)']
data_rainfall = cleancsv['Precipitation Rate']
data_bay_temp = cleancsv['Bay Temp']
data_salinity = cleancsv['Salinity']
data_lbi_temp = cleancsv['LBI Air Temp']
data_ocean_temp = cleancsv['Ocean Temp']
data_onshore_flag = cleancsv['Onshore']
data_upwelling_flag = cleancsv['upwelling_flag']

#saves all input data into one Numpy array
dataset = np.column_stack([
    data_main_air_temp.values,
    data_humidity_per.values,
    data_wind_direction.values,
    data_wind_speed.values,
    data_gusting.values,
    data_pressure.values,
    data_rainfall.values,
    data_bay_temp.values,
    data_salinity.values,
    data_lbi_temp.values,
    data_ocean_temp.values,
    #data_onshore_flag.values,
    data_upwelling_flag.values,
])

#Save output data into variables and reshape it to be a 2d array
output_data = data_onshore_flag.values
output_data = np.array(output_data).reshape(-1, 1)

In [56]:
#Length of training data
training_data_len = int(np.ceil(len(dataset) * 0.90)) #Use 90% of training data

In [57]:
#Scaler
scaler_x= StandardScaler()
scaler_y= StandardScaler()

scaledx = scaler_x.fit_transform(dataset)
scaledy = scaler_y.fit_transform(output_data)

training_data_x = scaledx[:training_data_len] #90% of all data
training_data_y = output_data[:training_data_len] #90% of all data

X_train, y_train = [], []

In [58]:
#Sliding window over last 24 hrs
for i in range(24, training_data_len):
    X_train.append(training_data_x[i-24:i, :])
    y_train.append(training_data_y[i, 0])

#Convert lists to arrays
X_train = np.array(X_train)
y_train = np.array(y_train).reshape(-1, 1) #1D Array

In [59]:
test_x = scaledx[training_data_len-24:]
X_test = []

#rebuild window
for i in range(24, len(test_x)):
    X_test.append(test_x[i-24:i, :])

X_test = np.array(X_test)   # (samples_test, 24, n_features)
print(X_test.shape)


(218, 24, 12)


In [60]:
#Build the model
model = keras.models.Sequential()

In [61]:
#Layer Zero input_shape=(X_train.shape[1], 1)
model.add(keras.layers.Input(shape=(X_train.shape[1], X_train.shape[2])))

In [62]:
#First Layer input_shape=(X_train.shape[1], 1)
#model.add(keras.layers.LSTM(64, return_sequences=True))

In [63]:
#Second Layer
model.add(keras.layers.LSTM(32, return_sequences=False))

In [64]:
#3rd Layer (Dense)
model.add(keras.layers.Dense(32, activation="relu"))

In [65]:
#4th Layer (Dropout)
#model.add(keras.layers.Dropout(0.5))

In [66]:
#Final Output Layer (Dense)
model.add(keras.layers.Dense(1))

In [67]:
#Put all the layers together
model.summary()
model.compile(optimizer="adam",
              loss="mae",
              metrics=[keras.metrics.RootMeanSquaredError()])

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 32)             │         5,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,849 (26.75 KB)

 Trainable params: 6,849 (26.75 KB)

 Non-trainable params: 0 (0.00 B)

In [68]:
#Train the model

#epochs = # of runs
#batch size = how much data is in each batch
training = model.fit(
X_train,
    y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.1,
    shuffle=True
)

Epoch 1/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0730 - root_mean_squared_error: 0.1088 - val_loss: 0.0881 - val_root_mean_squared_error: 0.1036
Epoch 2/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0252 - root_mean_squared_error: 0.0339 - val_loss: 0.0650 - val_root_mean_squared_error: 0.0804
Epoch 3/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0178 - root_mean_squared_error: 0.0240 - val_loss: 0.0546 - val_root_mean_squared_error: 0.0628
Epoch 4/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0156 - root_mean_squared_error: 0.0206 - val_loss: 0.0422 - val_root_mean_squared_error: 0.0508
Epoch 5/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0127 - root_mean_squared_error: 0.0166 - val_loss: 0.0376 - val_root_mean_squared_error: 0.0456
Epoch 6/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0119 - root_mean_squared_error: 0.0157 - val_loss: 0.0318 - val_root_mean_squared_error: 0.0375
Epoch 7/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0101

In [69]:
prediction_scaled = model.predict(X_test)

# back to original units
prediction = scaler_y.inverse_transform(prediction_scaled)  


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


In [70]:
# rows of the original dataframe that correspond to X_test / prediction
test_index_start = training_data_len
test_index_end = training_data_len + prediction.shape[0]

test_df = cleancsv.iloc[test_index_start:test_index_end].copy()

# add predicted column
test_df["wind_pred"] = prediction.ravel()

test_df.to_csv("CSV/predictions.csv", index=False)


In [71]:
# true wind speed for the same rows as predictions
y_test_true = cleancsv['Onshore'].iloc[test_index_start:test_index_end].values

mae = mean_absolute_error(y_test_true, prediction.ravel())
print("Test MAE (m/s):", mae)

Test MAE (m/s): 0.016904808580875397


In [72]:
print(prediction_scaled.shape)
print(prediction_scaled[:10].ravel())
print(prediction_scaled.mean(), prediction_scaled.std())

(218, 1)
[-0.00969743 -0.0113825  -0.01875822 -0.02392736 -0.02477111 -0.02368063
 -0.0218076  -0.02229915 -0.01970864 -0.01601717]
-0.007020247 0.01883835


In [73]:

training.history['val_loss']

[0.0881458967924118,
 0.06496869772672653,
 0.05459874868392944,
 0.04222598299384117,
 0.0376317985355854,
 0.03177928552031517,
 0.030831273645162582,
 0.02718661166727543,
 0.02759367786347866,
 0.030834000557661057,
 0.025041110813617706,
 0.028609804809093475,
 0.024128176271915436,
 0.023747382685542107,
 0.023372376337647438,
 0.021755514666438103,
 0.018746979534626007,
 0.020258331671357155,
 0.018528161570429802,
 0.018390994518995285,
 0.018797431141138077,
 0.016678355634212494,
 0.013855050317943096,
 0.016029704362154007,
 0.01338296290487051,
 0.013620523735880852,
 0.01404481939971447,
 0.015687912702560425,
 0.010543995536863804,
 0.012758525088429451,
 0.012371129356324673,
 0.011619442142546177,
 0.012724918313324451,
 0.012257995083928108,
 0.01156274601817131,
 0.010717521421611309,
 0.00955235306173563,
 0.010469882749021053,
 0.010708343237638474,
 0.011089774779975414]

In [74]:
print("Wind speed mean/std:", output_data.mean(), output_data.std())
print("First 10 wind speeds:", output_data[:10].ravel())

Wind speed mean/std: 0.0 0.0
First 10 wind speeds: [0 0 0 0 0 0 0 0 0 0]


In [75]:
i0 = training_data_len   # first test index
print("Example window X[0] (first test sample):")
print(cleancsv.iloc[i0-24:i0][["Wind Speed (A)", "Mainland Air Temp", "Direction (A)"]].head(10))
print("Target at i0:", cleancsv["Wind Speed (A)"].iloc[i0])

Example window X[0] (first test sample):
      Wind Speed (A)  Mainland Air Temp  Direction (A)
1942             4.2               19.6              0
1943             4.6               18.3              0
1944             4.0               17.1             15
1945             3.9               16.1             15
1946             4.3               15.2             15
1947             3.8               14.7             15
1948             4.7               14.6             15
1949             5.7               14.1             15
1950             5.7               14.3             15
1951             6.1               14.9             15
Target at i0: 3.0
